# 01 Foundation Model Loading Smoke Test

This notebook checks the selected real foundation model checkpoints:

- Prithvi-EO-2.0 300M
- TerraMind-1.0 Base
- TerraMind-1.0 Large

By default it verifies Hugging Face access, config/code files, and weight-file presence without downloading multi-GB checkpoints. On a GPU VM, set `LOAD_MODEL_WEIGHTS=1` before launching Jupyter to download and open the checkpoint files.


In [1]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
PROJECT_ROOT

PosixPath('/Users/similoluwaokunowo/Desktop/Research/geoai-lightning-talk-research')

In [2]:
import pandas as pd

from eval_harness.config import load_yaml, model_config_path
from eval_harness.models.foundation import smoke_check_foundation_model as check_foundation_model

model_names = ['prithvi_eo_v2_300m', 'terramind_v1_base', 'terramind_v1_large']
model_configs = {name: load_yaml(model_config_path(name)) for name in model_names}

pd.DataFrame([
    {
        'model': cfg['model'],
        'display_name': cfg['display_name'],
        'hf_repo': cfg['hf_repo'],
        'checkpoint_file': cfg['checkpoint_file'],
        'parameter_count_estimate': cfg['parameter_count_estimate'],
        'machine_type': cfg['machine_type'],
        'precision': cfg['precision'],
        'role': cfg['selection_role'],
    }
    for cfg in model_configs.values()
])


,model,display_name,hf_repo,checkpoint_file,parameter_count_estimate,machine_type,precision,role
0,prithvi_eo_v2_300m,Prithvi-EO-2.0 300M,ibm-nasa-geospatial/Prithvi-EO-2.0-300M,Prithvi_EO_V2_300M.pt,300000000,a2-highgpu-1g,bf16,primary
1,terramind_v1_base,TerraMind-1.0 Base,ibm-esa-geospatial/TerraMind-1.0-base,TerraMind_v1_base.pt,380000000,a2-highgpu-1g,bf16,second_model
2,terramind_v1_large,TerraMind-1.0 Large,ibm-esa-geospatial/TerraMind-1.0-large,TerraMind_v1_large.pt,950000000,a2-ultragpu-1g,bf16,scaling_stress_test


In [3]:
load_weights = os.environ.get('LOAD_MODEL_WEIGHTS') == '1'
print('LOAD_MODEL_WEIGHTS =', load_weights)

results = []
for name, cfg in model_configs.items():
    print(f'\nMODEL: {name}')
    result = check_foundation_model(cfg, load_weights=load_weights).as_dict()
    results.append(result)
    print('repo:', result['hf_repo'])
    print('access_ok:', result['access_ok'])
    print('weight_files:', result['weight_files'])
    print('checkpoint_load_ok:', result['checkpoint_load_ok'])
    if result['checkpoint_summary']:
        print('checkpoint_summary:', result['checkpoint_summary'])

model_results = pd.DataFrame(results)
display(model_results[[
    'model',
    'display_name',
    'hf_repo',
    'is_foundation_model',
    'access_ok',
    'parameter_count_estimate',
    'checkpoint_size_estimate',
    'load_weights',
    'checkpoint_load_ok',
]])


LOAD_MODEL_WEIGHTS = False

MODEL: prithvi_eo_v2_300m
repo: ibm-nasa-geospatial/Prithvi-EO-2.0-300M
access_ok: True
weight_files: ['Prithvi_EO_V2_300M.pt']
checkpoint_load_ok: None

MODEL: terramind_v1_base
repo: ibm-esa-geospatial/TerraMind-1.0-base
access_ok: True
weight_files: ['TerraMind_v1_base.pt']
checkpoint_load_ok: None

MODEL: terramind_v1_large
repo: ibm-esa-geospatial/TerraMind-1.0-large
access_ok: True
weight_files: ['TerraMind_v1_large.pt']
checkpoint_load_ok: None


,model,display_name,hf_repo,is_foundation_model,access_ok,parameter_count_estimate,checkpoint_size_estimate,load_weights,checkpoint_load_ok
0,prithvi_eo_v2_300m,Prithvi-EO-2.0 300M,ibm-nasa-geospatial/Prithvi-EO-2.0-300M,True,True,300000000,unknown,False,None
1,terramind_v1_base,TerraMind-1.0 Base,ibm-esa-geospatial/TerraMind-1.0-base,True,True,380000000,1.52GB,False,None
2,terramind_v1_large,TerraMind-1.0 Large,ibm-esa-geospatial/TerraMind-1.0-large,True,True,950000000,3.79GB,False,None


In [4]:
for result in results:
    print(f"\n{result['model']}")
    print('config/code files:')
    for file in result['config_files'][:12]:
        print(' -', file)
    print('downloaded local files:')
    for file in result['local_files'][:12]:
        print(' -', file)



prithvi_eo_v2_300m
config/code files:
 - README.md
 - config.json
 - inference.py
 - prithvi_mae.py
 - requirements.txt
downloaded local files:
 - /Users/similoluwaokunowo/.cache/huggingface/hub/models--ibm-nasa-geospatial--Prithvi-EO-2.0-300M/snapshots/9eb1b1102806593963daa333bcc491b1c6f8562f/README.md
 - /Users/similoluwaokunowo/.cache/huggingface/hub/models--ibm-nasa-geospatial--Prithvi-EO-2.0-300M/snapshots/9eb1b1102806593963daa333bcc491b1c6f8562f/config.json
 - /Users/similoluwaokunowo/.cache/huggingface/hub/models--ibm-nasa-geospatial--Prithvi-EO-2.0-300M/snapshots/9eb1b1102806593963daa333bcc491b1c6f8562f/inference.py
 - /Users/similoluwaokunowo/.cache/huggingface/hub/models--ibm-nasa-geospatial--Prithvi-EO-2.0-300M/snapshots/9eb1b1102806593963daa333bcc491b1c6f8562f/prithvi_mae.py
 - /Users/similoluwaokunowo/.cache/huggingface/hub/models--ibm-nasa-geospatial--Prithvi-EO-2.0-300M/snapshots/9eb1b1102806593963daa333bcc491b1c6f8562f/requirements.txt

terramind_v1_base
config/code fi

## Pass Criteria

- All three selected Hugging Face repos are reachable.
- Each repo exposes the expected checkpoint file.
- Metadata/config files download locally.
- With `LOAD_MODEL_WEIGHTS=1`, the checkpoint file downloads and `torch.load(..., map_location='cpu')` can inspect it.

This is real checkpoint loading. Architecture-specific forward passes and LoRA target-module selection are the next integration step after data EDA.
